In [2]:
import pandas as pd
import joblib
import sys

sys.path.append('..')
from src.features.blocking import make_block_keys
from src.pipeline.search import search_top_k

In [3]:
# ============================================================
# LOAD DATA AND MODEL
# ============================================================

df = pd.read_parquet(
    "../data/processed/processed_dataset.parquet"
)

model = joblib.load(
    "../data/artifacts/random_forest_matcher.joblib"
)

print(f"Dataset rows: {len(df):,}")
print("Model loaded.")

Dataset rows: 3,653,581
Model loaded.


In [4]:
# ============================================================
# PREPARE SEARCH INDEX
# ============================================================

search_index = make_block_keys(
    df.copy(),
    name_col="name_latin",
)

print(f"Search index rows: {len(search_index):,}")

display(
    search_index[
        [
            "party_name",
            "name_latin",
            "block_prefix_len",
            "block_first_token_len",
            "block_sorted_tokens",
        ]
    ].head()
)

Search index rows: 3,653,581


,party_name,name_latin,block_prefix_len,block_first_token_len,block_sorted_tokens
0,Անդրանիկ Աբարյան,andranik abaryan,andr_3,andr_3,abaryan andranik
1,Տիգրան Հարությունյան,tigran haroutyounyan,tigr_4,tigr_4,haroutyounyan tigran
2,Նարեկ Նարգիզյան,narek nargizyan,nare_3,nare_3,narek nargizyan
3,ՎԱՍԿԵՆ ՅԱԿՈՒԲՅԱՆ,vasken yakoubyan,vask_3,vask_3,vasken yakoubyan
4,Ժիրայր Ավանյան,zhirayr avanyan,zhir_3,zhir_3,avanyan zhirayr


In [5]:
# ============================================================
# SEARCH EXAMPLE 1: TYPO
# ============================================================

query = {
    "party_name": "Шурупов Александр Александрович",
    "country": "kgz",
    "relation_kind": "director",
    "relation_role": "director",
}

results = search_top_k(
    record=query,
    database_df=search_index,
    model=model,
    top_k=10,
)

display(results)

,record_id_2,party_name_2,name_latin_2,country_2,relation_kind_2,relation_role_2,company_public_id_2,company_name_norm_2,similarity_score
0,kgz_rec_99a3d95d4f2d344b4ad80eaf0cc9d4fe,Шурупов Александр Алексанрович,shurupov aleksandr aleksanrovich,kgz,director,director,kgz_company_af0872b424c70fca,каратор ресурстар жоопкерчилиги чектелген коому,0.999999
1,kgz_rec_58f08eb31283e5d17a385af279329210,Шурупов Александр Алексанрович,shurupov aleksandr aleksanrovich,kgz,shareholder,founder,kgz_company_af0872b424c70fca,каратор ресурстар жоопкерчилиги чектелген коому,0.999999
2,kgz_rec_8152d50114a74b79ae090e2e27ebf6a9,Шурупов Александр Алексанрович,shurupov aleksandr aleksanrovich,kgz,shareholder,founder,kgz_company_af0872b424c70fca,каратор ресурстар жоопкерчилиги чектелген коому,0.999999
3,kgz_rec_d153fc8285c06023fc433019365a0041,Шурупов Александр Александрович,shurupov aleksandr aleksandrovich,kgz,shareholder,founder,kgz_company_9fc11a4c79a57ef7,j l company джей энд эль компани жоопкерчилиги...,0.999988
4,kgz_rec_b6aef266f086562a7029594fb8b09975,Шурупов Александр Александрович,shurupov aleksandr aleksandrovich,kgz,shareholder,founder,kgz_company_b133483de9cc9d49,tsb temp system building тиэсби тэмп систем би...,0.999988
5,kgz_rec_c30f302ea4e105ecd5f5183dffb56a41,Шурупов Александр Александрович,shurupov aleksandr aleksandrovich,kgz,shareholder,founder,kgz_company_0684b5232b15972f,каратор ресурсы жабык акционердик коому,0.999988
6,kgz_rec_6bfc5e8bd3751a0c763a6f1560a81690,Шурупов Александр Александрович,shurupov aleksandr aleksandrovich,kgz,shareholder,founder,kgz_company_d3c46369b64e4dfb,аспара плэйсер жоопкерчилиги чектелген коому,0.999988
7,kgz_rec_5973915f9ed0aaafe6ec275470abc65d,Шурупов Александр Александрович,shurupov aleksandr aleksandrovich,kgz,shareholder,founder,kgz_company_b133483de9cc9d49,tsb temp system building тиэсби тэмп систем би...,0.999988
8,kgz_rec_fa4ee3e6b8a653dfee60ebd1fbfb7ead,Шурупов Александр Александрович,shurupov aleksandr aleksandrovich,kgz,shareholder,founder,kgz_company_822c5783437d0615,киндервуд жоопкерчилиги чектелген коому,0.999988
9,kgz_rec_56b77418242f959ade22549f57624458,Шурупов Александр Александрович,shurupov aleksandr aleksandrovich,kgz,shareholder,founder,kgz_company_42f1fec49d2475bd,алтын сай жоопкерчилиги чектелген коому,0.999988


In [6]:
# ============================================================
# SEARCH EXAMPLE 2: TOKEN ORDER
# ============================================================

query = {
    "party_name": "Yufei Wang",
    "country": "mng",
}

results = search_top_k(
    record=query,
    database_df=search_index,
    model=model,
    top_k=10,
)

display(results)

,record_id_2,party_name_2,name_latin_2,country_2,relation_kind_2,relation_role_2,company_public_id_2,company_name_norm_2,similarity_score
0,mng_rec_b65d8dd88ebdbecb4a3ab1793416e0cb,Wang Yufei,wang yufei,mng,shareholder,shareholder,mng_company_683141910dcb77b1,хар алт эрдэнэ майнинг,0.999992
1,mng_rec_fcd343fcc10d9696dd86f4d74283bc73,. Wang Yufei,wang yufei,mng,shareholder,shareholder,mng_company_22b385780d5a2f7f,сод мандал майнинг,0.999992
2,mng_rec_b7feb73619b7e5bda3e41fac7a76aebd,- Wang Yufei,wang yufei,mng,shareholder,shareholder,mng_company_0a17e2496d7cf9dc,их мандал хүрд ресурс проспектинг,0.999992
3,mng_rec_55322e34afc0990ea5f73f523e0acf4c,Wang Yufei,wang yufei,mng,director,Гүйцэтгэх захирал,mng_company_37c8f13d263fc09b,сод мандал эрдэнэ майнинг,0.999992
4,mng_rec_14ec2744fc58284820354987fa274592,WANG YUFEI,wang yufei,mng,shareholder,shareholder,mng_company_df9839f771f20f3e,мэнгуванши групп,0.999992
5,mng_rec_f6b7e7474a27671fde1c1a9f83c34e31,Wang Yufei,wang yufei,mng,shareholder,shareholder,mng_company_d1642a52443d8ca1,хар алт эрдэнэ коал интернэшнл,0.999992
6,mng_rec_569813a15691c1b4367a940d84f055e8,Wang Yufei,wang yufei,mng,shareholder,shareholder,mng_company_47eb2886768c9f9b,мега стар майнинг,0.999992
7,mng_rec_1829f64aa5eb348ced63e03b22fe9c26,wang yufei,wang yufei,mng,shareholder,shareholder,mng_company_b217da4618e14729,зуун уул,0.999992
8,mng_rec_9860ab039edbf0a4c9604c6d3762082e,Wang Yufei,wang yufei,mng,shareholder,shareholder,mng_company_695d1791da8a9726,хар алт эрдэнэ ложистик,0.999992
9,mng_rec_9209a98fd9e8d0c17e31a74cb5d8fa0a,WANG YUFEI,wang yufei,mng,shareholder,shareholder,mng_company_d8859a00989cb759,алтан ар,0.999992


In [7]:
# ============================================================
# SEARCH EXAMPLE 3: COMPANY
# ============================================================

query = {
    "party_name": "Neo Metals Holding Limited",
    "country": "arm",
    "relation_kind": "shareholder",
}

results = search_top_k(
    record=query,
    database_df=search_index,
    model=model,
    top_k=10,
)

display(results)

,record_id_2,party_name_2,name_latin_2,country_2,relation_kind_2,relation_role_2,company_public_id_2,company_name_norm_2,similarity_score
0,arm_rec_46d7e66aa9d6abdba98485f7877b7dd3,ՆԵՈ ՄԵՏԱԼՍ ՀՈԼԴԻՆԳ ԼԻՄԻԹԵԴ,neo metals holding limited,arm,shareholder,beneficial_owner,arm_company_4ac77af48cd1504b,արդյունաբերական ընկերություն բը,0.99993
1,arm_rec_db2bcc00a30c5c6e9fe93e9087c5881f,ՆԵՈ ՄԵՏԱԼՍ ՀՈԼԴԻՆԳ ԼԻՄԻԹԵԴ,neo metals holding limited,arm,shareholder,beneficial_owner,arm_company_4ac77af48cd1504b,պրոմիշլեննայա կոմպանիա,0.99993
2,arm_rec_6f4cc1b3898f630b466f6ff57f1bdcee,ՆԵՈ ՄԵՏԱԼՍ ՀՈԼԴԻՆԳ ԼԻՄԻԹԵԴ,neo metals holding limited,arm,shareholder,beneficial_owner,arm_company_4ac77af48cd1504b,արդյունաբերական ընկերություն բը,0.99993
3,arm_rec_58581fdf2c8f910a3996860984a4fed6,ՆԵՈ ՄԵՏԱԼՍ ՀՈԼԴԻՆԳ ԼԻՄԻԹԵԴ,neo metals holding limited,arm,shareholder,beneficial_owner,arm_company_e803ae9e12061f1f,լեռ էքս,0.99993
4,arm_rec_4ee16f7e033738836cc69c0f466a8d0f,ՆԵՈ ՄԵՏԱԼՍ ՀՈԼԴԻՆԳ ԼԻՄԻԹԵԴ,neo metals holding limited,arm,shareholder,beneficial_owner,arm_company_4ac77af48cd1504b,արդյունաբերական ընկերություն բը,0.99993
5,arm_rec_a51eb617f4848801f7c5d69c7bf90e64,ՆԵՈ ՄԵՏԱԼՍ ՀՈԼԴԻՆԳ ԼԻՄԻԹԵԴ,neo metals holding limited,arm,shareholder,beneficial_owner,arm_company_4ac77af48cd1504b,արդյունաբերական ընկերություն բը,0.99993
6,arm_rec_46d7e66aa9d6abdba98485f7877b7dd3,ՆԵՈ ՄԵՏԱԼՍ ՀՈԼԴԻՆԳ ԼԻՄԻԹԵԴ,neo metals holding limited,arm,shareholder,beneficial_owner,arm_company_4ac77af48cd1504b,պրոմիշլեննայա կոմպանիա,0.99993
7,arm_rec_db2bcc00a30c5c6e9fe93e9087c5881f,ՆԵՈ ՄԵՏԱԼՍ ՀՈԼԴԻՆԳ ԼԻՄԻԹԵԴ,neo metals holding limited,arm,shareholder,beneficial_owner,arm_company_4ac77af48cd1504b,արդյունաբերական ընկերություն բը,0.99993
8,arm_rec_46d7e66aa9d6abdba98485f7877b7dd3,ՆԵՈ ՄԵՏԱԼՍ ՀՈԼԴԻՆԳ ԼԻՄԻԹԵԴ,neo metals holding limited,arm,shareholder,beneficial_owner,arm_company_4ac77af48cd1504b,արդյունաբերական ընկերություն բը,0.99993
9,arm_rec_6f4cc1b3898f630b466f6ff57f1bdcee,ՆԵՈ ՄԵՏԱԼՍ ՀՈԼԴԻՆԳ ԼԻՄԻԹԵԴ,neo metals holding limited,arm,shareholder,beneficial_owner,arm_company_4ac77af48cd1504b,արդյունաբերական ընկերություն բը,0.99993


In [8]:
# ============================================================
# SEARCH EXAMPLE 4: ARMENIAN / TRANSLITERATION
# ============================================================

query = {
    "party_name": "ՄՈՐԻՑ ՋՈՆԱԹԱՆ ՀԻԵՐՈՆԻՄՈՒՍ ՀԻԼԼ",
    "country": "arm",
}

results = search_top_k(
    record=query,
    database_df=search_index,
    model=model,
    top_k=10,
)

display(results)

,record_id_2,party_name_2,name_latin_2,country_2,relation_kind_2,relation_role_2,company_public_id_2,company_name_norm_2,similarity_score
0,arm_rec_798ed9e56f7a13ec2365ba01fbb18463,ՄՈՐԻՑ ՋՀՈՆԱԹԱՆ ՀԻԵՐՈՆԻՄՈՒՍ ՀԻԼԼ,morits jhonatan hieronimous hill,arm,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ,0.999999
1,arm_rec_27e542367fa07f2f59a2efbc675c7bc6,ՄՈՐԻՑ ՋՀՈՆԱԹԱՆ ՀԻԵՐՈՆԻՄՈՒՍ ՀԻԼԼ,morits jhonatan hieronimous hill,arm,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ,0.999999
2,arm_rec_9ebf05dde33952f38d0f3b4e78a0cc09,ՄՈՐԻՑ ՋՀՈՆԱԹԱՆ ՀԻԵՐՈՆԻՄՈՒՍ ՀԻԼԼ,morits jhonatan hieronimous hill,arm,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ,0.999999
3,arm_rec_37fa3edb58621ba0cc7fdd37918deab5,ՄՈՐԻՑ ՋՀՈՆԱԹԱՆ ՀԻԵՐՈՆԻՄՈՒՍ ՀԻԼԼ,morits jhonatan hieronimous hill,arm,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ,0.999999
4,arm_rec_0760bfff8fe6d269cfbc255482d459b1,ՄՈՐԻՑ ՋՀՈՆԱԹԱՆ ՀԻԵՐՈՆԻՄՈՒՍ ՀԻԼԼ,morits jhonatan hieronimous hill,arm,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ,0.999999
5,arm_rec_bb8d6f6edc9a71eee4bcb5fc84366ed2,ՄՈՐԻՑ ՋՀՈՆԱԹԱՆ ՀԻԵՐՈՆԻՄՈՒՍ ՀԻԼԼ,morits jhonatan hieronimous hill,arm,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ,0.999999
6,arm_rec_5c1f764104ec5233f01daf579d4fd3cd,ՄՈՐԻՑ ՋՀՈՆԱԹԱՆ ՀԻԵՐՈՆԻՄՈՒՍ ՀԻԼԼ,morits jhonatan hieronimous hill,arm,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ,0.999999
7,arm_rec_cbcd538560a86ef0774f83442fa35533,ՄՈՐԻՑ ՋՀՈՆԱԹԱՆ ՀԻԵՐՈՆԻՄՈՒՍ ՀԻԼԼ,morits jhonatan hieronimous hill,arm,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ,0.999999
8,arm_rec_27e542367fa07f2f59a2efbc675c7bc6,ՄՈՐԻՑ ՋՀՈՆԱԹԱՆ ՀԻԵՐՈՆԻՄՈՒՍ ՀԻԼԼ,morits jhonatan hieronimous hill,arm,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ,0.999999
9,arm_rec_5e2f7ef1ffde1732457e331318838371,ՄՈՐԻՑ ՋՀՈՆԱԹԱՆ ՀԻԵՐՈՆԻՄՈՒՍ ՀԻԼԼ,morits jhonatan hieronimous hill,arm,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ,0.999999


In [9]:
# ============================================================
# SEARCH EMPTY / NO MATCH CASE
# ============================================================

query = {
    "party_name": "Completely Unknown Random Person Xyzabc",
    "country": "arm",
}

results = search_top_k(
    record=query,
    database_df=search_index,
    model=model,
    top_k=10,
)

display(results)

""


In [10]:
# ============================================================
# SAVE SEARCH INDEX SAMPLE
# ============================================================

search_index.head(10000).to_parquet(
    "../data/processed/search_index_sample.parquet",
    index=False,
)

print("Saved search index sample.")

Saved search index sample.


**Вывод:** Search pipeline успешно находит релевантные записи по новой входящей записи. Поиск поддерживает опечатки, перестановку токенов, мультиязычные записи и транслитерацию. На текущем этапе поиск возвращает наиболее похожие записи из базы. Для production API дополнительно добавлена дедупликация выдачи, чтобы top-K содержал уникальные сущности, а не повторяющиеся записи одной и той же сущности.